# Diffusion & Flow Exercise


## 1 Diffusion is not like a flow with a fixed rate

In diffusion, molecules move randomly due to thermal motion.
There is no fixed “flow rate” like in a pipe.
The longer the distance a molecule has to travel, the longer it will typically take to get there.
So, “all molecules will eventually diffuse anyway” is true — but the time it takes grows with the square of the distance. That’s why diffusion is only fast over short distances.

The **characteristic distance** is essentially the **typical distance over which something diffuses** — the distance you care about for your problem. It’s not “every molecule travels exactly this distance”. It’s the **scale at which diffusion matters**.
Think of it as the **length scale of your system**.

1. **Inside a cell**
   * A cell might be 10 µm across.
   * If oxygen has to get from the cell membrane to the center, the **characteristic distance** is ~10 µm.
2. **Tissue**
   * A layer of tissue might be 1 mm thick.
   * Oxygen has to diffuse from the nearest blood vessel to the middle of the tissue.
   * Then the characteristic distance is ~0.5 mm (half the thickness, because on average molecules start at the edge and diffuse to the center).
3. **Diffusion in air**
   * If you spray perfume in a room, the characteristic distance could be meters — the scale over which you’re asking “how long does it take for molecules to spread?”

## What $x$ really means
$x$ is a **characteristic distance over which you care about diffusion**. For example: 
If you have a cell 10 µm thick, $x \sim 10, \mu m$ is the distance oxygen needs to travel from the capillary to the center of the cell. 
9 If you have a 1 mm tissue, $x \sim 1, mm$ is the distance oxygen needs to reach the middle.
The diffusion timescale formula:
$$
t \sim \frac{x^2}{2D}
$$
So diffusion works for **local transport**, but it cannot efficiently supply large distances quickly. Intuitively, you can think like this:
For **tiny distances (µm)** diffusion is extremely fast — basically instantaneous for cells.
For **millimeter distances** diffusion is slow — which is why tissues thicker than ~0.1–0.2 mm need **blood vessels** to bring oxygen in.

## **Exercise: Diffusion Timescale**
Compute timescale for diffusion across different tissue thicknesses, with lengths of 1 µm and 1 mm with $D = 1.8\cdot 10^{-3} mm^2/s$ for $O_2$-diffusion coefficient.
* What is the timescales for the two different tissue thicknesses 1 µm and 1 mm?
* Assume a timescale greater than 2 ms is too large for effecient diffusion of $O_2$, if only considering diffusion, what thickness is diffusion no longer viable?

In [ ]:
import math

D = 0.0018  # mm^2/s, same as 1.8e-5 cm^2s^-1
x1 = 0.001
x2 = 1

# Solution

## Fick + Convection (analytic steady 1D solution)


If we know the boundary conditions, the equation for Concentration (C) can be calculated using the following equation:
$$ C(x) = C_0 + (C_L - C_0)\frac{e^{v x / D} - 1}{e^{v L / D} - 1} $$



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define parameters
C0 = 1.0  # mol/mm^3
CL = 0.1  # mol/mm^3
L = 0.1  # mm
v = 0.1  # mm/s
D = 1.8e-3  # mm^2/s

# Create position array from 0 to L
x = np.linspace(0.01 * L, L, 100)

# Compute concentration profile using analytical solution
C = C0 + (CL - C0) * np.expm1(v * x / D) / np.expm1(v * L / D)

# Plot the concentration profile
plt.figure(figsize=(8, 5))
plt.plot(x * 1e3, C, label="C(x)")
plt.xlabel("Tissue thickness (μm)")
plt.ylabel("Concentration [mol/mm³]")
plt.title("Oxygen Concentration Profile Across Tissue")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()



However, if we do not know the boundary conditions, or if the flow is diffusion dominated, we may use the differential equation $-D \frac{d^2C}{dx^2} + v \frac{dC}{dx} = 0$, which integrates into an exponential decay model, where $\lambda_{eff} = \sqrt{2 D \Delta t}$ and $\Delta t = \frac{L_{axial}}{v}$. 

$$ C(x) = C_0 * e^{\frac{-x}{\lambda_{eff}}} $$



### Flux 
We can use the steady 1D convection–diffusion equation to calculate flux:
    
$$ 
J_s = -D \frac{\partial C(x,t)}{\partial x} + J_v C(x,t) 
$$



### Peclet's Number
The Péclet number (Pe) is a dimensionless quantity representing the ratio of transport by convection (advection) to transport by diffusion, and for oxygen transfer from blood to a cell, it can be generally expressed as 
$$Pe = \frac{\text{convective transport​}}{\text{diffusive transport}}= \frac{v * L}{D}$$

Here:
* `v` is the characteristic fluid velocity (e.g., red blood cell velocity or plasma flow velocity)
* `L` is a characteristic length scale (such as the capillary radius or cell size)
* `D` is the diffusion coefficient of oxygen in the surrounding medium (e.g., plasma or tissue)

A high `Pe` value, where convection dominates, means oxygen is quickly transported by the flow, while a low value indicates diffusion is the primary transport mechanism


**Interpretation for Oxygen Transfer**
- **Pe > 1 (Convection-Dominated):**
    When the Peclet number is significantly greater than 1, it signifies that the rate at which oxygen is transported by the fluid's bulk flow (convection) is much faster than its transport due to concentration gradients (diffusion). In this regime, the movement of oxygen is primarily dictated by the fluid's movement. 
- **Pe < 1 (Diffusion-Dominated):** 
    If the Peclet number is less than 1, diffusion becomes the dominant process for oxygen transport. The movement of oxygen is driven by its tendency to spread from areas of high concentration to areas of low concentration. 
- **Pe ≈ 1 (Mixed Regimes):**
    In situations where the Peclet number is around 1, both convection and diffusion are important and contribute to the overall transport of oxygen. 

In [ ]:
from files.fluxmodel import concentration, flux, peclet
# add conditions to simulate different scenarios in the variable above!
conditions = {
    "Normal": {"v": 0.1,"D": 1.8e-3},  # Normal Scenario
    "Valsalva":{"v":0.3,"D":1.8e-3}, # Valsalva Maneuver
    "Fibrosis": { "v": 0.1,"D": 0.9e-3},  # Fibrosis Scenario
    "Fibrosis+Valsalva": { "v": 0.3,"D": 0.9e-3}  # Fibrosis Scenario
     }


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.axes

L_axial = 1.0  # mm
L = 0.1  # mm
C0 = 1.0  # mol/mm^3
x = np.linspace(0, L, 101)  # mm





# Create subplots
fig, axs = plt.subplots(1, 3, figsize=(18, 8))
axs: list[matplotlib.axes.Axes]

for i, (label, params) in enumerate(conditions.items()):
    v = params["v"]
    D = params["D"]

    # Flux components
    C = concentration(x, L_axial, v, D, C0)
    Js = flux(x, C, v, D)
    Pe = peclet(x, v, D)

    # Plot fluxes
    axs[0].plot(x, Js, label=label)

    # Plot concentration
    axs[1].plot(x , C, label=label)

    # Plot peclet
    axs[2].plot(x, Pe, label=label)

# Set titles and labels
axs[0].set_title("Flux")
axs[0].set_xlabel("Tissue barrier thickness (μm)")
axs[0].set_ylabel("Flux [mol/mm²/s]")
axs[0].set_xscale("log")
# axs[0].set_yscale("log")
axs[0].grid(True)
axs[0].legend()
axs[1].set_title("Concentration")
axs[1].set_xlabel("Tissue barrier thickness (μm)")
axs[1].set_ylabel("Concentration [mol/mm³]")
axs[1].set_xscale("log")
# axs[1].set_yscale("log")
axs[1].grid(True)
axs[1].legend()
lbl = "Diff./Conv. Barrier"
axs[2].plot(x,np.ones_like(x),label=lbl,linestyle="--",color="k")
axs[2].set_title("Peclet's Number")
axs[2].set_xlabel("Tissue barrier thickness (μm)")
axs[2].set_ylabel("Peclet Number")
axs[2].set_xscale("log")
axs[2].set_yscale("log")
axs[2].grid(True)
axs[2].legend()
plt.tight_layout()
plt.show()


## Physiological Scenarios in the Convection-Diffusion Model

The model is used to simulate solute transport across tissue barriers includes two key parameters:

- **v**: velocity of fluid flow (e.g., interstitial or capillary flow)
- **D**: diffusion coefficient of the solute (e.g., oxygen)

These parameters determine how solute concentration changes across the tissue. We are now going to simulate two scenarios.
1. Valsalva Maneuver
2. Fibrosis

### Valsalva Maneuver

**Valsalva** is a forced exhalation against a closed airway, often used in clinical tests. It increases **intrathoracic pressure**, which can affect blood and interstitial flow.
In this task, you will simulate the **Valsalva Maneuver**, a physiological condition that increases intrathoracic pressure and fluid velocity.
- **Effect on $v$**: ↑ Increased velocity due to pressure-driven flow
- **Effect on $D$**: No direct change, but tissue compression may slightly alter diffusion



### Fibrosis
**Fibrosis** is the thickening and scarring of connective tissue, often due to chronic inflammation or disease.
- **Effect on  D**: ↓ Reduced diffusion due to dense extracellular matrix
- **Effect on  v**: May be unchanged or slightly reduced due to restricted flow


### Valsalva + Fibrosis
-  $v$ : ↑ Increased
-  $D$ : ↓ Decreased





## Exercise: Conditional effect on Concentration

1. Extend the model to include three new conditions: `valsalva`, `fibrosis`, `valsalva+fibrosis`:

|| v | D |
|---|---|---|
|Normal| 0.1 | 1.8e-3 |
|Valsalva| 0.3 | 1.8e-3 |
|Fibrosis| 0.1 | 0.9e-3 |
|Valsalva + Fibrosis| 0.3 | 0.9e-3 |

*Note, in the code above, you can add scenarios, as elements in the `conditions` dictionary variable.*



2. What happens to the solute transport in each case? `valsalva`, `fibrosis`, `valsalva+fibrosis` 
    - Effect on Flux?
    - Effect on Concentration?
    - What happens to Peclets number, and what does that mean?
    - Explain the overall effect of the intervention




## Answer

#### Valsalva

#### Fibrosis

#### Valsalva + Fibrosis


###  Summary Table

| Condition           |  v  |  D  | Concentration Profile | Flux Behavior | Oxygen Delivery |
|---------------------|--------|--------|------------------------|----------------|------------------|
| Normal              | 0.1      | 1.8e-3 | Moderate decay         | Balanced       | Effective        |
| Valsalva            | 0.3      | 1.8e-3 |           |    |  |
| Fibrosis            | 0.1      | 0.9e-3 |           |      |          |
| Valsalva + Fibrosis | 0.3      | 0.9e-3 |       |  |   |

